<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 36px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>00. Gaussian Mixture Models with TFP</b>
</div>

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">📑 Table of Contents</span>

- **1. [What Are Mixture Models?](#-part-i-what-are-mixture-models)**
- **2. [Setup & Imports](#-part-ii-setup--imports)**
- **3. [GMMs with TFP: MixtureSameFamily](#-part-iii-gmms-with-tfp)**
    - 3.1 [Building a 1D Gaussian Mixture](#building-a-1d-gaussian-mixture)
    - 3.2 [Sampling and Density Evaluation](#sampling-and-density-evaluation)
    - 3.3 [2D Gaussian Mixtures](#2d-gaussian-mixtures)
- **4. [Fitting GMMs to Data](#-part-iv-fitting-gmms-to-data)**
    - 4.1 [Maximum Likelihood via Gradient Descent](#ml-gradient-descent)
    - 4.2 [Visualizing the Fit](#visualizing-the-fit)
- **5. [Model Selection: How Many Components?](#-part-v-model-selection)**
- **6. [Applications of GMMs](#-part-vi-applications)**
    - 6.1 [Clustering](#clustering)
    - 6.2 [Density Estimation](#density-estimation)
- **7. [Key Takeaways](#-part-vii-key-takeaways)**

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">1. What Are Mixture Models?</span>

A **mixture model** represents a probability distribution as a weighted combination of simpler distributions:

$$p(x) = \sum_{k=1}^{K} \pi_k \cdot \mathcal{N}(x | \mu_k, \Sigma_k)$$

Where:
- $K$ = number of components
- $\pi_k$ = mixing weights ($\sum_k \pi_k = 1$)
- $\mu_k, \Sigma_k$ = mean and covariance of component $k$

| **Use Case** | **Why GMMs?** |
|-------------|---------------|
| Clustering | Soft assignments with uncertainty |
| Density estimation | Approximate any continuous distribution |
| Anomaly detection | Low-density regions indicate anomalies |
| Feature extraction | Latent cluster assignments as features |
| Generative modeling | Sample new data from the learned distribution |

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">2. Setup & Imports</span>

In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib import cm

tfd = tfp.distributions
tfb = tfp.bijectors

tf.random.set_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print(f"TensorFlow: {tf.__version__}")
print(f"TFP: {tfp.__version__}")

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">3. GMMs with TFP: MixtureSameFamily</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.1. Building a 1D Gaussian Mixture</span>

In [ ]:
# Create a 1D Gaussian Mixture Model with 3 components
gmm_1d = tfd.MixtureSameFamily(
    mixture_distribution=tfd.Categorical(probs=[0.3, 0.5, 0.2]),
    components_distribution=tfd.Normal(
        loc=[-3.0, 1.0, 5.0],      # Component means
        scale=[0.8, 1.2, 0.5]       # Component standard deviations
    )
)

print("GMM 1D:")
print(f"  Batch shape: {gmm_1d.batch_shape}")
print(f"  Event shape: {gmm_1d.event_shape}")
print(f"  Mean: {gmm_1d.mean().numpy():.3f}")

# Visualize
x = np.linspace(-7, 8, 500).astype(np.float32)
density = gmm_1d.prob(x).numpy()

# Individual components
comp_probs = [0.3, 0.5, 0.2]
comp_means = [-3.0, 1.0, 5.0]
comp_stds = [0.8, 1.2, 0.5]
colors = ['#ef4444', '#3b82f6', '#22c55e']

fig, ax = plt.subplots(figsize=(14, 6))

ax.fill_between(x, density, alpha=0.3, color='#8b5cf6')
ax.plot(x, density, color='#8b5cf6', linewidth=2.5, label='Mixture')

for i, (p, m, s, c) in enumerate(zip(comp_probs, comp_means, comp_stds, colors)):
    comp = tfd.Normal(loc=m, scale=s)
    ax.plot(x, p * comp.prob(x).numpy(), '--', color=c, linewidth=2, 
            label=f'Component {i+1}: π={p}, μ={m}, σ={s}', alpha=0.8)

ax.set_xlabel('x', fontsize=14)
ax.set_ylabel('Density', fontsize=14)
ax.set_title('1D Gaussian Mixture Model (3 Components)', fontsize=16, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.2. Sampling and Density Evaluation</span>

In [ ]:
# Sample from the GMM
samples = gmm_1d.sample(5000)

# Log probability evaluation
test_points = tf.constant([-3.0, 0.0, 1.0, 5.0])
log_probs = gmm_1d.log_prob(test_points)

print("Log probabilities at test points:")
for x_val, lp in zip(test_points.numpy(), log_probs.numpy()):
    print(f"  x = {x_val:5.1f} → log p(x) = {lp:.4f}, p(x) = {np.exp(lp):.4f}")

# Histogram of samples vs true density
fig, ax = plt.subplots(figsize=(14, 6))
ax.hist(samples.numpy(), bins=100, density=True, alpha=0.5, color='#3b82f6', label='Samples (n=5000)')
x_range = np.linspace(-7, 8, 500).astype(np.float32)
ax.plot(x_range, gmm_1d.prob(x_range).numpy(), color='#ef4444', linewidth=2.5, label='True density')
ax.set_xlabel('x', fontsize=14)
ax.set_ylabel('Density', fontsize=14)
ax.set_title('GMM Samples vs True Density', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">3.3. 2D Gaussian Mixtures</span>

In [ ]:
# 2D GMM with 4 components
gmm_2d = tfd.MixtureSameFamily(
    mixture_distribution=tfd.Categorical(probs=[0.25, 0.25, 0.25, 0.25]),
    components_distribution=tfd.MultivariateNormalDiag(
        loc=[[-3., -3.], [3., 3.], [-3., 3.], [3., -3.]],
        scale_diag=[[0.8, 0.8], [1.0, 0.5], [0.5, 1.0], [0.7, 0.7]]
    )
)

# Sample and visualize
samples_2d = gmm_2d.sample(3000).numpy()

# Create density grid
xx, yy = np.meshgrid(np.linspace(-6, 6, 200), np.linspace(-6, 6, 200))
grid = np.stack([xx.ravel(), yy.ravel()], axis=-1).astype(np.float32)
density_2d = gmm_2d.prob(grid).numpy().reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Scatter plot
axes[0].scatter(samples_2d[:, 0], samples_2d[:, 1], alpha=0.3, s=5, c='#8b5cf6')
axes[0].set_title('Samples from 2D GMM', fontsize=15, fontweight='bold')
axes[0].set_xlabel('$x_1$', fontsize=13)
axes[0].set_ylabel('$x_2$', fontsize=13)

# Contour plot
axes[1].contourf(xx, yy, density_2d, levels=30, cmap='viridis')
axes[1].contour(xx, yy, density_2d, levels=10, colors='white', alpha=0.3, linewidths=0.5)
axes[1].set_title('Density Contours', fontsize=15, fontweight='bold')
axes[1].set_xlabel('$x_1$', fontsize=13)
axes[1].set_ylabel('$x_2$', fontsize=13)

plt.suptitle('2D Gaussian Mixture Model (4 Components)', fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">4. Fitting GMMs to Data</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">4.1. Maximum Likelihood via Gradient Descent</span>

In [ ]:
# Generate training data from a known mixture
true_gmm = tfd.MixtureSameFamily(
    mixture_distribution=tfd.Categorical(probs=[0.4, 0.6]),
    components_distribution=tfd.Normal(loc=[-2.0, 3.0], scale=[0.7, 1.0])
)
train_data = true_gmm.sample(1000)

# Trainable GMM parameters
num_components = 2
logits = tf.Variable(tf.zeros(num_components), name='logits')  # Mixing weights (unconstrained)
locs = tf.Variable(tf.random.normal([num_components]), name='locs')
log_scales = tf.Variable(tf.zeros(num_components), name='log_scales')  # Log scale (positive via exp)

def make_gmm():
    return tfd.MixtureSameFamily(
        mixture_distribution=tfd.Categorical(logits=logits),
        components_distribution=tfd.Normal(
            loc=locs,
            scale=tf.nn.softplus(log_scales) + 1e-5
        )
    )

# Training loop
optimizer = tf.keras.optimizers.Adam(learning_rate=0.05)
losses = []

for step in range(500):
    with tf.GradientTape() as tape:
        gmm = make_gmm()
        nll = -tf.reduce_mean(gmm.log_prob(train_data))  # Negative log-likelihood
    
    grads = tape.gradient(nll, [logits, locs, log_scales])
    optimizer.apply_gradients(zip(grads, [logits, locs, log_scales]))
    losses.append(nll.numpy())
    
    if step % 100 == 0:
        probs = tf.nn.softmax(logits).numpy()
        scales = (tf.nn.softplus(log_scales) + 1e-5).numpy()
        print(f"Step {step:3d}: NLL={nll:.4f}, "
              f"weights={probs}, means={locs.numpy()}, scales={scales}")

# Final parameters
print(f"\n{'='*60}")
print("Learned vs True Parameters:")
print(f"  Weights: {tf.nn.softmax(logits).numpy()} (true: [0.4, 0.6])")
print(f"  Means:   {locs.numpy()} (true: [-2.0, 3.0])")
print(f"  Scales:  {(tf.nn.softplus(log_scales) + 1e-5).numpy()} (true: [0.7, 1.0])")

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">4.2. Visualizing the Fit</span>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Training curve
axes[0].plot(losses, color='#3b82f6', linewidth=2)
axes[0].set_xlabel('Step', fontsize=13)
axes[0].set_ylabel('Negative Log-Likelihood', fontsize=13)
axes[0].set_title('Training Curve', fontsize=15, fontweight='bold')

# Learned vs true density
x_plot = np.linspace(-6, 7, 500).astype(np.float32)
learned_gmm = make_gmm()

axes[1].hist(train_data.numpy(), bins=60, density=True, alpha=0.4, color='#94a3b8', label='Data')
axes[1].plot(x_plot, true_gmm.prob(x_plot).numpy(), 'r--', linewidth=2.5, label='True density')
axes[1].plot(x_plot, learned_gmm.prob(x_plot).numpy(), color='#3b82f6', linewidth=2.5, label='Learned density')
axes[1].set_xlabel('x', fontsize=13)
axes[1].set_ylabel('Density', fontsize=13)
axes[1].set_title('Learned GMM vs True Distribution', fontsize=15, fontweight='bold')
axes[1].legend(fontsize=12)

plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">5. Model Selection: How Many Components?</span>

In [ ]:
def fit_gmm(data, K, num_steps=400):
    """Fit a K-component GMM via MLE and return final NLL and BIC."""
    logits_k = tf.Variable(tf.zeros(K))
    locs_k = tf.Variable(tf.random.normal([K]) * 3)
    log_scales_k = tf.Variable(tf.zeros(K))
    
    opt = tf.keras.optimizers.Adam(0.05)
    
    for _ in range(num_steps):
        with tf.GradientTape() as tape:
            gmm_k = tfd.MixtureSameFamily(
                mixture_distribution=tfd.Categorical(logits=logits_k),
                components_distribution=tfd.Normal(
                    loc=locs_k, scale=tf.nn.softplus(log_scales_k) + 1e-5
                )
            )
            nll = -tf.reduce_mean(gmm_k.log_prob(data))
        grads = tape.gradient(nll, [logits_k, locs_k, log_scales_k])
        opt.apply_gradients(zip(grads, [logits_k, locs_k, log_scales_k]))
    
    n = len(data)
    num_params = 3 * K - 1  # K means + K scales + (K-1) mixing weights
    log_lik = -nll.numpy() * n
    bic = num_params * np.log(n) - 2 * log_lik
    aic = 2 * num_params - 2 * log_lik
    
    return nll.numpy(), bic, aic

# Compare K = 1 to 6
ks = range(1, 7)
results = []
for k in ks:
    nll, bic, aic = fit_gmm(train_data, k)
    results.append({'K': k, 'NLL': nll, 'BIC': bic, 'AIC': aic})
    print(f"K={k}: NLL={nll:.4f}, BIC={bic:.1f}, AIC={aic:.1f}")

best_k_bic = min(results, key=lambda r: r['BIC'])['K']
print(f"\n🏆 Best K (by BIC): {best_k_bic}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

bics = [r['BIC'] for r in results]
aics = [r['AIC'] for r in results]

ax.plot(list(ks), bics, 'o-', color='#3b82f6', linewidth=2, markersize=8, label='BIC')
ax.plot(list(ks), aics, 's-', color='#ef4444', linewidth=2, markersize=8, label='AIC')
ax.axvline(x=best_k_bic, color='#22c55e', linestyle='--', linewidth=2, label=f'Best K={best_k_bic}')

ax.set_xlabel('Number of Components (K)', fontsize=14)
ax.set_ylabel('Information Criterion', fontsize=14)
ax.set_title('GMM Model Selection: BIC and AIC', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.set_xticks(list(ks))
plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">6. Applications of GMMs</span>

### <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #312e81, #111827); padding: 10px 18px; border-radius: 10px; font-size: 24px; font-weight: 600; box-shadow: 0 3px 10px rgba(0,0,0,0.15);">6.1. Soft Clustering with GMMs</span>

In [ ]:
# Generate 2D clustered data
np.random.seed(42)
cluster_data = np.vstack([
    np.random.multivariate_normal([-3, -2], [[0.5, 0.2], [0.2, 0.5]], 150),
    np.random.multivariate_normal([2, 3], [[0.8, -0.3], [-0.3, 0.6]], 200),
    np.random.multivariate_normal([4, -1], [[0.3, 0], [0, 0.9]], 100),
]).astype(np.float32)

# Fit a 3-component 2D GMM
K = 3
logits_2d = tf.Variable(tf.zeros(K))
locs_2d = tf.Variable(tf.random.normal([K, 2]) * 2)
log_scales_2d = tf.Variable(tf.zeros([K, 2]))

optimizer = tf.keras.optimizers.Adam(0.03)

for step in range(600):
    with tf.GradientTape() as tape:
        gmm_fit = tfd.MixtureSameFamily(
            mixture_distribution=tfd.Categorical(logits=logits_2d),
            components_distribution=tfd.MultivariateNormalDiag(
                loc=locs_2d,
                scale_diag=tf.nn.softplus(log_scales_2d) + 1e-5
            )
        )
        loss = -tf.reduce_mean(gmm_fit.log_prob(cluster_data))
    grads = tape.gradient(loss, [logits_2d, locs_2d, log_scales_2d])
    optimizer.apply_gradients(zip(grads, [logits_2d, locs_2d, log_scales_2d]))

# Compute soft assignments (responsibilities)
component_log_probs = gmm_fit.components_distribution.log_prob(cluster_data[:, tf.newaxis, :])
log_weights = tf.nn.log_softmax(logits_2d)
log_responsibilities = component_log_probs + log_weights
responsibilities = tf.nn.softmax(log_responsibilities, axis=1).numpy()
hard_assignments = np.argmax(responsibilities, axis=1)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
colors_map = ['#ef4444', '#3b82f6', '#22c55e']

# Hard assignments
for k in range(K):
    mask = hard_assignments == k
    axes[0].scatter(cluster_data[mask, 0], cluster_data[mask, 1], 
                    c=colors_map[k], s=20, alpha=0.6, label=f'Cluster {k+1}')
axes[0].scatter(locs_2d.numpy()[:, 0], locs_2d.numpy()[:, 1], 
                c='black', marker='X', s=200, zorder=5, edgecolors='white', linewidth=2)
axes[0].set_title('Hard Clustering', fontsize=15, fontweight='bold')
axes[0].legend(fontsize=11)

# Soft assignments (color by responsibility)
scatter = axes[1].scatter(cluster_data[:, 0], cluster_data[:, 1], 
                          c=responsibilities[:, 0], cmap='coolwarm', s=20, alpha=0.7)
plt.colorbar(scatter, ax=axes[1], label='P(cluster 1 | x)')
axes[1].set_title('Soft Clustering (Responsibilities)', fontsize=15, fontweight='bold')

for ax in axes:
    ax.set_xlabel('$x_1$', fontsize=13)
    ax.set_ylabel('$x_2$', fontsize=13)

plt.tight_layout()
plt.show()

## <span style="display: inline-block; color: #fff; background: linear-gradient(135deg, #dc2626, #0ea5e9); padding: 12px 20px; border-radius: 12px; font-size: 28px; font-weight: 700; box-shadow: 0 4px 12px rgba(0,0,0,0.2);">7. Key Takeaways</span>

| **Concept** | **Key Point** |
|------------|---------------|
| `MixtureSameFamily` | TFP's built-in class for mixture models |
| Trainable GMM | Use `tf.Variable` parameters + gradient-based optimization |
| Softplus trick | Ensures positive scale parameters: `tf.nn.softplus(x)` |
| BIC/AIC | Select the number of components K |
| Soft clustering | Probabilistic assignments give uncertainty over clusters |

### Next Steps
- **Notebook 01**: Learn the **EM algorithm** — the classic approach to fitting GMMs
- **Notebook 02**: Explore **Bayesian Mixture Models** with Dirichlet priors and MCMC inference